In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

In [2]:
class AnalizadorPrerrequisitos:
    def __init__(self):
        self.df_prereq = None
        self.df_historial = None
        self.resultado = None
        self.max_niveles_cadena = 5  # Máximo número de niveles en la cadena de prerrequisitos
    
    def cargar_documentos(self):
        """Solicita y carga los documentos de Excel"""
        print("=== ANALIZADOR DE PRERREQUISITOS ACADÉMICOS CON CADENAS ===\n")
        
        # Solicitar archivo de prerrequisitos
        while True:
            try:
                ruta_prereq = "para trabajar pre req progv22025-09-15095342_procesado.xlsx" #input("Ingrese la ruta del archivo 'pre_requisitos.xlsx': ").strip()
                self.df_prereq = pd.read_excel(ruta_prereq)
                print(f"✓ Archivo de prerrequisitos cargado: {len(self.df_prereq)} registros")
                break
            except Exception as e:
                print(f"Error al cargar prerrequisitos: {e}")
                print("Intente nuevamente.\n")
        
        # Solicitar archivo de historial
        while True:
            try:
                ruta_historial = "detalle matricula sistemas pedacito.xlsx" #input("Ingrese la ruta del archivo 'historial_asignaturas.xlsx': ").strip()
                self.df_historial = pd.read_excel(ruta_historial)
                print(f"✓ Archivo de historial cargado: {len(self.df_historial)} registros")
                break
            except Exception as e:
                print(f"Error al cargar historial: {e}")
                print("Intente nuevamente.\n")
    
    def validar_columnas(self):
        """Valida que existan las columnas necesarias"""
        # Columnas requeridas en prerrequisitos
        cols_prereq_req = ['Smbarul_Key_Rule', 'Programa']
        cols_prereq_faltantes = [col for col in cols_prereq_req if col not in self.df_prereq.columns]
        
        if cols_prereq_faltantes:
            raise ValueError(f"Faltan columnas en prerrequisitos: {cols_prereq_faltantes}")
        
        # Columnas requeridas en historial
        cols_hist_req = ['Codigo_Programa', 'Cod materia curso', 'Pidm', 'Materia_Aprobada', 'Calificacion_Final']
        cols_hist_faltantes = [col for col in cols_hist_req if col not in self.df_historial.columns]
        
        if cols_hist_faltantes:
            raise ValueError(f"Faltan columnas en historial: {cols_hist_faltantes}")
        
        print("✓ Todas las columnas necesarias están presentes")
    
    def obtener_columnas_opciones(self):
        """Obtiene las columnas de opciones de prerrequisitos (Opcion_Prereq_1 a Opcion_Prereq_21)"""
        patron = r'^Opcion_Prereq_\d+$'
        columnas_opciones = [col for col in self.df_prereq.columns if re.match(patron, col)]
        columnas_opciones.sort(key=lambda x: int(x.split('_')[-1]))  # Ordenar numéricamente
        return columnas_opciones
    
    def parsear_prerrequisito(self, prereq_str):
        """
        Parsea una cadena de prerrequisito y devuelve una lista de códigos de asignaturas
        Ejemplos:
        - "IGL4990" -> ["IGL4990"]
        - "DIG0020 & HUM4040" -> ["DIG0020", "HUM4040"]
        """
        if pd.isna(prereq_str) or str(prereq_str).strip() == '':
            return []
        
        prereq_str = str(prereq_str).strip()
        
        # Si contiene '&', son múltiples prerrequisitos
        if '&' in prereq_str:
            codigos = [codigo.strip() for codigo in prereq_str.split('&')]
            return [codigo for codigo in codigos if codigo]  # Filtrar vacíos
        else:
            return [prereq_str] if prereq_str else []
    
    def obtener_info_materia_estudiante(self, pidm, codigo_materia):
        """
        Obtiene información detallada de una materia para un estudiante específico
        Retorna: dict con información de la materia
        """
        historial_estudiante = self.df_historial[
            (self.df_historial['Pidm'] == pidm) & 
            (self.df_historial['Cod materia curso'] == codigo_materia)
        ]
        
        if historial_estudiante.empty:
            return {
                'codigo': codigo_materia,
                'encontrada': False,
                'aprobada': False,
                'nota_final': None,
                'intentos': 0,
                'todas_notas': []
            }
        
        # Contar intentos
        intentos = len(historial_estudiante)
        
        # Obtener todas las notas
        todas_notas = historial_estudiante['Calificacion_Final'].tolist()
        
        # Verificar si alguna vez fue aprobada
        registros_aprobados = historial_estudiante[
            historial_estudiante['Materia_Aprobada'].astype(str).str.upper() == 'Y'
        ]
        
        if not registros_aprobados.empty:
            # Tomar el primer registro aprobado
            registro_aprobado = registros_aprobados.iloc[0]
            aprobada = True
            nota_final = registro_aprobado['Calificacion_Final']
        else:
            # No fue aprobada, tomar el último intento
            ultimo_registro = historial_estudiante.iloc[-1]
            aprobada = False
            nota_final = ultimo_registro['Calificacion_Final']
        
        return {
            'codigo': codigo_materia,
            'encontrada': True,
            'aprobada': aprobada,
            'nota_final': nota_final,
            'intentos': intentos,
            'todas_notas': todas_notas
        }
    
    def obtener_prerrequisitos_materia(self, codigo_materia):
        """
        Obtiene los prerrequisitos de una materia específica
        Retorna: lista de códigos de prerrequisitos de la primera opción válida
        """
        prereq_info = self.df_prereq[
            self.df_prereq['Smbarul_Key_Rule'] == codigo_materia
        ]
        
        if prereq_info.empty:
            return []
        
        columnas_opciones = self.obtener_columnas_opciones()
        prereq_info_row = prereq_info.iloc[0]
        
        # Buscar la primera opción no vacía
        for col_opcion in columnas_opciones:
            opcion_value = prereq_info_row[col_opcion]
            
            if pd.isna(opcion_value) or str(opcion_value).strip() == '':
                continue
            
            codigos_prereq = self.parsear_prerrequisito(opcion_value)
            if codigos_prereq:
                return codigos_prereq
        
        return []
    
    def construir_cadena_prerrequisitos(self, pidm, codigo_materia, nivel=0, visitados=None):
        """
        Construye recursivamente la cadena completa de prerrequisitos
        Retorna: dict con la estructura completa de prerrequisitos
        """
        if visitados is None:
            visitados = set()
        
        # Evitar ciclos infinitos
        if codigo_materia in visitados or nivel >= self.max_niveles_cadena:
            return None
        
        visitados.add(codigo_materia)
        
        # Obtener información de la materia para este estudiante
        info_materia = self.obtener_info_materia_estudiante(pidm, codigo_materia)
        
        # Obtener prerrequisitos de esta materia
        prerrequisitos_codigos = self.obtener_prerrequisitos_materia(codigo_materia)
        
        # Construir información de prerrequisitos recursivamente
        prerrequisitos_info = []
        for prereq_codigo in prerrequisitos_codigos:
            prereq_info = self.construir_cadena_prerrequisitos(
                pidm, prereq_codigo, nivel + 1, visitados.copy()
            )
            if prereq_info:
                prerrequisitos_info.append(prereq_info)
        
        return {
            'codigo': codigo_materia,
            'info_estudiante': info_materia,
            'prerrequisitos': prerrequisitos_info,
            'nivel': nivel
        }
    
    def verificar_prerrequisitos_estudiante(self, pidm, codigos_prereq):
        """
        Verifica si un estudiante ha aprobado todos los prerrequisitos de una lista
        Retorna: (cumple: bool, detalles: list)
        """
        if not codigos_prereq:
            return True, []
        
        detalles = []
        
        for codigo in codigos_prereq:
            info_materia = self.obtener_info_materia_estudiante(pidm, codigo)
            detalles.append(info_materia)
        
        # Todos deben estar aprobados para que se cumpla la opción
        cumple = all(detalle['aprobada'] for detalle in detalles)
        return cumple, detalles
    
    def crear_columnas_resultado(self):
        """Crea las columnas para el DataFrame resultado"""
        columnas_base = list(self.df_historial.columns)
        
        # Columnas de resumen
        columnas_base.extend([
            'Prerrequisito_Estado',
            'Num_Prerrequisitos_Directos',
            'Intentos_Materia_Principal'
        ])
        
        # Determinar el número máximo de prerrequisitos directos
        max_prereq_directos = 5  # Asumiendo máximo 5 prerrequisitos directos
        
        # Columnas para prerrequisitos directos
        for i in range(1, max_prereq_directos + 1):
            columnas_base.extend([
                f'Prereq_{i}_Codigo',
                f'Prereq_{i}_Nota',
                f'Prereq_{i}_Intentos',
                f'Prereq_{i}_Estado'
            ])
        
        # Columnas para cadenas de prerrequisitos (hasta 5 niveles)
        for nivel in range(1, self.max_niveles_cadena + 1):
            for i in range(1, max_prereq_directos + 1):
                columnas_base.extend([
                    f'Cadena_N{nivel}_Prereq_{i}_Codigo',
                    f'Cadena_N{nivel}_Prereq_{i}_Nota',
                    f'Cadena_N{nivel}_Prereq_{i}_Intentos',
                    f'Cadena_N{nivel}_Prereq_{i}_Estado'
                ])
        
        return columnas_base
    
    def aplanar_cadena_prerrequisitos(self, cadena_info, resultado_dict, prefijo=""):
        """
        Aplana la estructura jerárquica de prerrequisitos en un diccionario plano
        """
        if not cadena_info or not cadena_info.get('prerrequisitos'):
            return
        
        prerrequisitos = cadena_info['prerrequisitos']
        
        # Procesar prerrequisitos del nivel actual
        for i, prereq in enumerate(prerrequisitos, 1):
            if i > 5:  # Máximo 5 prerrequisitos por nivel
                break
                
            info_est = prereq['info_estudiante']
            nivel = prereq['nivel']
            
            if nivel == 1:  # Prerrequisitos directos
                resultado_dict[f'Prereq_{i}_Codigo'] = info_est['codigo']
                resultado_dict[f'Prereq_{i}_Nota'] = info_est['nota_final']
                resultado_dict[f'Prereq_{i}_Intentos'] = info_est['intentos']
                resultado_dict[f'Prereq_{i}_Estado'] = 'Aprobada' if info_est['aprobada'] else 'No Aprobada'
            else:  # Prerrequisitos de la cadena
                resultado_dict[f'Cadena_N{nivel}_Prereq_{i}_Codigo'] = info_est['codigo']
                resultado_dict[f'Cadena_N{nivel}_Prereq_{i}_Nota'] = info_est['nota_final']
                resultado_dict[f'Cadena_N{nivel}_Prereq_{i}_Intentos'] = info_est['intentos']
                resultado_dict[f'Cadena_N{nivel}_Prereq_{i}_Estado'] = 'Aprobada' if info_est['aprobada'] else 'No Aprobada'
            
            # Procesar recursivamente los prerrequisitos de este prerrequisito
            self.aplanar_cadena_prerrequisitos(prereq, resultado_dict, prefijo)
    
    def procesar_prerrequisitos(self):
        """Procesa todos los registros y determina los prerrequisitos cumplidos"""
        print("\n=== PROCESANDO PRERREQUISITOS Y CADENAS ===")
        
        self.validar_columnas()
        columnas_opciones = self.obtener_columnas_opciones()
        
        print(f"Encontradas {len(columnas_opciones)} columnas de opciones de prerrequisitos")
        print("Procesando registros...")
        
        # Crear DataFrame resultado con todas las columnas necesarias
        columnas_resultado = self.crear_columnas_resultado()
        self.resultado = pd.DataFrame(columns=columnas_resultado)
        
        total_registros = len(self.df_historial)
        
        for idx, row in self.df_historial.iterrows():
            if idx % 500 == 0:
                print(f"Procesando registro {idx + 1}/{total_registros}")
            
            # Crear diccionario con datos base
            resultado_fila = dict(row)
            
            pidm = row['Pidm']
            cod_materia = row['Cod materia curso']
            
            # Obtener número de intentos de la materia principal
            info_principal = self.obtener_info_materia_estudiante(pidm, cod_materia)
            resultado_fila['Intentos_Materia_Principal'] = info_principal['intentos']
            
            # Buscar prerrequisitos para esta materia
            prereq_info = self.df_prereq[
                self.df_prereq['Smbarul_Key_Rule'] == cod_materia
            ]
            
            if prereq_info.empty:
                resultado_fila['Prerrequisito_Estado'] = 'No tiene prerrequisito'
                resultado_fila['Num_Prerrequisitos_Directos'] = 0
            else:
                # Construir cadena completa de prerrequisitos
                cadena_completa = self.construir_cadena_prerrequisitos(pidm, cod_materia)
                
                if cadena_completa and cadena_completa.get('prerrequisitos'):
                    prerrequisitos_directos = cadena_completa['prerrequisitos']
                    
                    # Verificar si cumple los prerrequisitos directos
                    codigos_directos = [p['info_estudiante']['codigo'] for p in prerrequisitos_directos]
                    cumple, _ = self.verificar_prerrequisitos_estudiante(pidm, codigos_directos)
                    
                    resultado_fila['Prerrequisito_Estado'] = 'Prerrequisito cumplido' if cumple else 'Tiene prerrequisito y no se encontró en la historia'
                    resultado_fila['Num_Prerrequisitos_Directos'] = len(prerrequisitos_directos)
                    
                    # Aplanar la cadena de prerrequisitos en columnas
                    self.aplanar_cadena_prerrequisitos(cadena_completa, resultado_fila)
                else:
                    resultado_fila['Prerrequisito_Estado'] = 'Tiene prerrequisito y no se encontró en la historia'
                    resultado_fila['Num_Prerrequisitos_Directos'] = 0
            
            # Agregar fila al resultado
            self.resultado = pd.concat([self.resultado, pd.DataFrame([resultado_fila])], ignore_index=True)
        
        print(f"✓ Procesamiento completado: {total_registros} registros analizados")
    
    def generar_reporte(self):
        """Genera un reporte resumen del análisis"""
        if self.resultado is None:
            print("No hay resultados para reportar")
            return
        
        print("\n=== REPORTE DE RESULTADOS ===")
        
        resumen = self.resultado['Prerrequisito_Estado'].value_counts()
        print("\nResumen por estado:")
        for estado, cantidad in resumen.items():
            porcentaje = (cantidad / len(self.resultado)) * 100
            print(f"- {estado}: {cantidad} ({porcentaje:.1f}%)")
        
        # Estadísticas de prerrequisitos
        print(f"\nEstadísticas de prerrequisitos directos:")
        prereq_stats = self.resultado['Num_Prerrequisitos_Directos'].value_counts().sort_index()
        for num, cantidad in prereq_stats.items():
            print(f"- {num} prerrequisitos: {cantidad} materias")
        
        # Estadísticas de intentos
        intentos_promedio = self.resultado['Intentos_Materia_Principal'].mean()
        print(f"\nPromedio de intentos por materia: {intentos_promedio:.2f}")
        
        print(f"\nTotal de registros analizados: {len(self.resultado)}")
    
    def guardar_resultados(self):
        """Guarda los resultados en un archivo Excel"""
        if self.resultado is None:
            print("No hay resultados para guardar")
            return
        
        timestamp = pd.Timestamp.now().strftime("%Y-%m-%d%H%M%S")
        nombre_archivo = "resultados_prerrequisitos_cadenas_"+timestamp+".xlsx"
        
        try:
            # Limpiar columnas vacías para reducir el tamaño del archivo
            self.resultado = self.resultado.dropna(axis=1, how='all')
            
            self.resultado.to_excel(nombre_archivo, index=False)
            print(f"✓ Resultados guardados en: {nombre_archivo}")
            print(f"Columnas en el archivo: {len(self.resultado.columns)}")
        except Exception as e:
            print(f"Error al guardar resultados: {e}")
    
    def ejecutar(self):
        """Ejecuta el análisis completo"""
        try:
            self.cargar_documentos()
            self.procesar_prerrequisitos()
            self.generar_reporte()
            self.guardar_resultados()
            print("\n¡Análisis de cadenas de prerrequisitos completado exitosamente!")
        except Exception as e:
            print(f"\nError durante la ejecución: {e}")
            print("Verifique los archivos y vuelva a intentar.")

# Función principal
def main():
    analizador = AnalizadorPrerrequisitos()
    analizador.ejecutar()

if __name__ == "__main__":
    main()

=== ANALIZADOR DE PRERREQUISITOS ACADÉMICOS CON CADENAS ===

✓ Archivo de prerrequisitos cargado: 3101 registros
✓ Archivo de historial cargado: 28512 registros

=== PROCESANDO PRERREQUISITOS Y CADENAS ===
✓ Todas las columnas necesarias están presentes
Encontradas 21 columnas de opciones de prerrequisitos
Procesando registros...
Procesando registro 1/28512


C:\Users\vittorinor\AppData\Local\Temp\ipykernel_1356\3672045948.py:336: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.resultado = pd.concat([self.resultado, pd.DataFrame([resultado_fila])], ignore_index=True)
C:\Users\vittorinor\AppData\Local\Temp\ipykernel_1356\3672045948.py:336: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.resultado = pd.concat([self.resultado, pd.DataFrame([resultado_fila])], ignore_index=True)


Procesando registro 501/28512
Procesando registro 1001/28512
Procesando registro 1501/28512
Procesando registro 2001/28512
Procesando registro 2501/28512
Procesando registro 3001/28512
Procesando registro 3501/28512
Procesando registro 4001/28512
Procesando registro 4501/28512
Procesando registro 5001/28512
Procesando registro 5501/28512
Procesando registro 6001/28512
Procesando registro 6501/28512
Procesando registro 7001/28512
Procesando registro 7501/28512
Procesando registro 8001/28512
Procesando registro 8501/28512
Procesando registro 9001/28512
Procesando registro 9501/28512
Procesando registro 10001/28512
Procesando registro 10501/28512
Procesando registro 11001/28512
Procesando registro 11501/28512
Procesando registro 12001/28512
Procesando registro 12501/28512
Procesando registro 13001/28512
Procesando registro 13501/28512
Procesando registro 14001/28512
Procesando registro 14501/28512
Procesando registro 15001/28512
Procesando registro 15501/28512
Procesando registro 16001/28